<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 1 (Lunes) — Predicción — Regresión lineal**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 🏠 Regresión lineal simple: precio de casas en el condado de King

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Predecir el precio de venta de una casa (Seattle, 2014-2015) a partir de su área habitable. |
| 🔍 Comprensión de los datos | Revisamos `kc_house_data.csv`: tipos de variables, correlaciones y dispersión `price` vs. `sqft_living`. |
| 🧹 Preparación de los datos | Separamos entrenamiento/prueba, validamos rangos y aplicamos logaritmo para linealizar la relación. |
| 🤖 Modelado | Ajustamos `price ~ sqft_living` (en escala log-log) con mínimos cuadrados ordinarios (`statsmodels`). |
| 📊 Evaluación | Interpretamos el resumen OLS y revisamos residuos (histograma, caja y Q-Q) sobre entrenamiento y prueba. |
| 🚀 Despliegue y conclusiones | Estimamos el precio de una casa nueva y guardamos el modelo con `joblib` para reutilizarlo. |

> 🎯 **Analogía:** un tasador inmobiliario que aprende, a partir de miles de ventas, cuánto suma cada pie cuadrado al precio de una casa.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 👉 **1** | Lunes | **Predicción: regresión lineal y series de tiempo** |
| 2 | Martes | Clasificación 1: árboles de decisión, bosques aleatorios y KNN |
| 3 | Miércoles | Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales |
| 4 | Jueves | No supervisado: agrupación y asociación |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

### 📥 Datos del curso
Los archivos se descargan automáticamente (una sola vez) desde el repositorio del curso. Si ya los subió a `sample_data/`, no se vuelven a descargar.

In [ ]:
import os, urllib.request
REPO = 'https://gitlab.com/ejimenez/fiec_intro_machine_learning/-/raw/main'
os.makedirs('sample_data', exist_ok=True)
for nombre, dia in [('kc_house_data.csv', 'dia_1')]:
    ruta = f'sample_data/{nombre}'
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(f'{REPO}/{dia}/{nombre}', ruta)
        print('Descargado:', ruta)
    else:
        print('Ya existe:', ruta)

---
## 🎯 Fase 1 · Comprensión del negocio

Predecir el precio de venta de una casa (Seattle, 2014-2015) a partir de su área habitable.

📚 Lectura recomendada: [3]

## Análisis del Problema
Este conjunto de datos contiene precios de venta de casas para el Condado de King, que incluye Seattle. Incluye casas vendidas entre mayo de 2014 y mayo de 2015.

Es un gran conjunto de datos para evaluar modelos de regresión simples.

# Regresión Lineal Simple
Los modelos lineales son una de las herramientas más importantes del análisis cuantitativo. Los utilizamos cuando queremos predecir –o explicar– una variable dependiente a partir de una variable independiente. El uso mismo de la terminología de variables dependientes e independientes nos da una pista del abordaje: el ajuste de modelos lineales busca encontrar una función –en el sentido matemático– que nos permite expresar a una variable a partir de otras.

### Autores
   * Efrén Jiménez

###Librerias



In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import pearsonr
from scipy.stats import spearmanr
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
import joblib

import math


---
## 🔍 Fase 2 · Comprensión de los datos

Revisamos `kc_house_data.csv`: tipos de variables, correlaciones y dispersión `price` vs. `sqft_living`.

📚 Lectura recomendada: [4]

## Entendimiento de los Datos

Con el fin de analizar y tratar de predecir el precio de las casas ubicadas en el condado de King, se trabajó con un conjunto de datos que contiene 8273 observaciones y 2 variables:

- price (precio): numérica, con un rango de 115000 a 1950000.
- sqft_lot (metros cuadrados del lote): numérica, con un rango de 500 a 7730

### 🧬 Tipos de variables

En regresión, la **variable respuesta** (`price`) es numérica; las **predictoras** (`sqft_living`, …) pueden ser numéricas o categóricas. La matriz de dispersión ayuda a ver cuáles se relacionan con la respuesta.

![Matriz de diagramas de dispersión entre variables](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch10/images/10_03.png)
*Figura: Matriz de diagramas de dispersión entre variables. [7]*

In [ ]:
casas = pd.read_csv('sample_data/kc_house_data.csv', na_values='?')

print("----------------------------------------------summary---------------------------------------")
print(casas.info())

print("----------------------------------------------Correlación---------------------------------------")
print(casas.iloc[:,2:16].corr())

In [ ]:
# Calcula la matriz de correlación
corr_matrix = casas.iloc[:,2:16].corr()

# Crea un mapa de calor de la matriz de correlación
sns.heatmap(corr_matrix, cmap='coolwarm')

# Muestra la gráfica
plt.show()

**Librerías de gráficos**

1.   https://plotly.com/r/
2.   https://matplotlib.org/

In [ ]:
plt.scatter(casas['sqft_living'], casas['price'])
plt.xlabel('sqft_living')
plt.ylabel('price')
plt.show()

---
## 🧹 Fase 3 · Preparación de los datos

Separamos entrenamiento/prueba, validamos rangos y aplicamos logaritmo para linealizar la relación.

📚 Lectura recomendada: [4]

In [ ]:
# Cargar el archivo CSV en una variable llamada casas
casas = pd.read_csv('sample_data/kc_house_data.csv')

# Selecionar las columnas 3 y 6 del conjunto de datos
casas = casas[['sqft_living', 'price']]

# Ver la estructura del conjunto de datos
casas.info()

# Dividir el conjunto de datos en uno de entrenamiento y otro de pruebas
casas_entrenamiento, casas_prueba = train_test_split(casas, test_size=0.3, random_state=4)

>Es importante siempre validar los rangos de los conjuntos de datos creados, para evitar caer en extrapolación:

In [ ]:
print('----------------------------- Entrenamiento -----------------------------')
print(casas_entrenamiento.describe())

print('------------------------------- Pruebas ----------------------------------')
casas_prueba = casas_prueba[casas_prueba['sqft_living'] <= 7620]
print(casas_prueba.describe())

---
## 🔍 Fase 2 · Comprensión de los datos — relación entre variables en el conjunto de entrenamiento

Revisamos `kc_house_data.csv`: tipos de variables, correlaciones y dispersión `price` vs. `sqft_living`.

De acuerdo con los resúmenes anteriores, las observaciones estan en el rango esperado.
###### Se realizara un diagrama de dispersión

In [ ]:
# Calcular log(5000)
result = math.log(5000)
print(result)

In [ ]:
# Hacer la gráfica de dispersión con log(price) en el eje y y log(sqft_living) en el eje x
plt.scatter(x = np.log(casas_entrenamiento['sqft_living']), y = np.log(casas_entrenamiento['price']))

# Agregar título y etiquetas de los ejes
plt.title('Relación entre Precio y Metros cuadrados')
plt.xlabel('sqft_living')
plt.ylabel('price')

# Mostrar la gráfica
plt.show()

La información del gráfico anterior podemos complementarla con una matriz de correlación:

In [ ]:
# Hacer el histograma de sqft_living
plt.hist(casas['sqft_living'])

# Agregar título y etiquetas de los ejes
plt.title('Histograma de sqft_living')
plt.xlabel('sqft_living')
plt.ylabel('Frecuencia')

# Mostrar el histograma
plt.show()

In [ ]:
# Hacer el histograma de log(sqft_living)
plt.hist(np.log(casas_entrenamiento['sqft_living']))

# Agregar título y etiquetas de los ejes
plt.title('Histograma de log(sqft_living)')
plt.xlabel('log(sqft_living)')
plt.ylabel('Frecuencia')

# Mostrar el histograma
plt.show()

In [ ]:
# Hacer el histograma de log(sqft_living)
plt.hist(np.log(casas_entrenamiento['price']))

# Agregar título y etiquetas de los ejes
plt.title('Histograma de log(price)')
plt.xlabel('log(price)')
plt.ylabel('Frecuencia')

# Mostrar el histograma
plt.show()

In [ ]:
# Calcular el coeficiente de correlación para la columna "casas.entrenamiento"
correlacion =casas_entrenamiento.corr()

# Imprimir el resultado
print(correlacion)

### Tipos de relación
- -1,00 = Correlación negativa perfecta
- -0,90 = Correlación negativa muy fuerte
- -0,75 = Correlación negativa considerable
- -0,50 = Correlación negativa media
- -0,10 = Correlación negativa débil
- 0,00  = No existe correlación lineal alguna entre las variables
- 0,10  = Correlación positiva débil
- 0,50  = Correlación positiva media
- 0,75 = Correlación positiva considerable
- 0,90  = Correlación positiva muy fuerte
- 1,00  = Correlación positiva perfecta

### Exiten varios metodos de correlación
#### Coeficiente de Pearson

In [ ]:
# Calcular los logaritmos de las variables
log_price = np.log(casas['price'])
log_sqft_living = np.log(casas['sqft_living'])

# Calcular la correlación de Pearson
correlacion, p_value = pearsonr(log_price, log_sqft_living)

# Imprimir los resultados
print("Coeficiente de correlación de Pearson:", correlacion)
print("Valor p:", p_value)


#### Coeficiente de Spearman

In [ ]:
# Calcular la correlación de Spearman
correlacion, p_value = spearmanr(casas['price'], casas['sqft_living'])

# Imprimir los resultados
print("Coeficiente de correlación de Spearman:", correlacion)
print("Valor p:", p_value)

Como se puede apreciar en la matriz de gráficos de dispersión, y confirmar con la matriz de correlación, hay una correlación significativa entre los variables price y sqft_lot

---
## 🤖 Fase 4 · Modelado

Ajustamos `price ~ sqft_living` (en escala log-log) con mínimos cuadrados ordinarios (`statsmodels`).

📚 Lectura recomendada: [5]

Una vez seleccionadas las variables para incluir en el modelo de regresión, se procede a crearlo:

In [ ]:
# Ajustar el modelo de regresión lineal
reg_price = sm.OLS(np.log(casas_entrenamiento['price']), sm.add_constant(np.log(casas_entrenamiento['sqft_living']))).fit()

# Imprimir un resumen del modelo
print(reg_price.summary())

#### Nota:

La ecuación para el modelo de regresión lineal simple ajustado anteriormente es:

**price = b0 + b1 * sqft_living**

Donde:

price es la variable respuesta (el precio de la casa)
sqft_living es la variable predictora (el área habitable de la casa en pies cuadrados)

*   b0 es el intercepto o término constante del modelo
*   b1 es el coeficiente de la variable predictora (la pendiente de la línea de regresión)

**Ejemplo**

*   El intercepto (b0) es 69.253
*   El coeficiente de la variable predictora (b1) es 0.307

Entonces, la ecuación para el modelo sería:

**price = 69.253 + 0.307 * sqft_living**

Por ejemplo, si una casa tiene un área habitable de 2,000 pies cuadrados, la predicción del precio de esa casa sería:

**price = 69.253 + 0.307 * 2,000 = 695.813**

Por lo tanto, se estima que el precio de la casa sería de **$695,813.**





### Notas importantes

* **OLS Regression Results**: Es el título que muestra el tipo de modelo que se ha ajustado.
* **Dep. Variable**: Es la variable respuesta (en este caso, el precio de la casa).
* **Model**: Es el modelo de regresión lineal simple ajustado. Muestra que el precio se está modelando como una función de la variable sqft_living.
* **Method**: Es el método que se utilizó para ajustar el modelo (en este caso, el método utilizado fue Ordinary Least Squares o Mínimos Cuadrados Ordinarios).
* **No. Observations**: Es el número de observaciones en el conjunto de entrenamiento que se usó para ajustar el modelo.
* **DF Residuals**: Es el número de grados de libertad restantes después de ajustar el modelo.
* **DF Model**: Es el número de grados de libertad en el modelo.
* **R-squared**: Es el coeficiente de determinación, que mide la proporción de la variabilidad en la variable respuesta que se explica por la variable predictora. Un valor de 1 indica una relación perfecta, mientras que un valor de 0 indica que la variable predictora no explica nada de la variabilidad en la variable respuesta.
* **Adj. R-squared**: Es el coeficiente de determinación ajustado, que tiene en cuenta la complejidad del modelo (es decir, el número de variables predictoras) y penaliza el uso de variables innecesarias.
* **F-statistic**: Es la estadística F, que se utiliza para evaluar la significación global del modelo.
* **Prob (F-statistic)**: Es el valor p asociado con la estadística F.
* **Intercept**: Es el término constante del modelo, que representa el valor predicho de la variable respuesta cuando todas las variables predictoras son iguales a cero.
* **sqft_living**: Es el coeficiente de la variable predictora, que representa el cambio en la variable respuesta por unidad de cambio en la variable predictora.
* **std err**: Es el error estándar del coeficiente, que mide la variabilidad de los coeficientes en diferentes muestras.
* **t**: Es la estadística t, que se utiliza para evaluar la significación individual de cada variable predictora.
* **P > |t|**: Es el valor p asociado con la estadística t.
* **Confidence Interval**: Es el intervalo de confianza al 95% para el coeficiente, que muestra el rango probable de valores verdaderos del coeficiente en la población.

---
## 📊 Fase 5 · Evaluación

Interpretamos el resumen OLS y revisamos residuos (histograma, caja y Q-Q) sobre entrenamiento y prueba.

📚 Lectura recomendada: [4] [7]

Se realiza una evaluación de los residuos de la regresión lineal.

### Residuos: ¿qué buscamos?

Un buen modelo deja residuos sin estructura, centrados en 0. Una forma curva indica que falta capturar una relación no lineal.

![Residuos con patrón curvo: el modelo lineal no captura toda la relación](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch10/images/10_09.png)
*Figura: Residuos con patrón curvo: el modelo lineal no captura toda la relación. [7]*

![Residuos repartidos al azar alrededor de 0](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch10/images/10_15.png)
*Figura: Residuos repartidos al azar alrededor de 0. [7]*

In [ ]:
# Calcular residuos estandarizados
residuos = reg_price.get_influence().resid_studentized_internal

# Generar gráficos para visualizar residuos estandarizados
fig, ax = plt.subplots(1, 3, figsize=(12,4))
ax[0].hist(residuos)
ax[0].set_title('Histograma')
ax[1].boxplot(residuos)
ax[1].set_title('Diagrama de cajas')
sm.qqplot(residuos, line='s', ax=ax[2])
ax[2].set_title('Gráfico de cuantiles')
plt.show()

    Se realizará un gráfico para observar la línea del modelo de regresión lineal

In [ ]:
# Transformar las variables a logaritmos
y = np.log(casas_entrenamiento['price'])
x = np.log(casas_entrenamiento['sqft_living'])

# Añadir una constante al modelo
#x = sm.add_constant(x)

# Graficar datos y modelo de regresión lineal
plt.scatter(x, y)
plt.plot(x, reg_price.predict(), color='red' , label="Recta",linestyle='-')
plt.xlabel('sqft_living')
plt.ylabel('price')
plt.show()

En el siguiente gráfico, se muestra la línea del modelo de regresión lineal y los coeficientes que se generaron en color rojo para reflejar la distancia correspondiente entre la línea de predicción y el valor real de la observación.

In [ ]:
# Definir función de predicción
def predt(modelo, x):
  return modelo.params.iloc[0] + x * modelo.params.iloc[1]

# Graficar datos y modelo de regresión lineal
plt.scatter(x, y, alpha=0.5)
plt.plot(x, reg_price.predict(), color='red')
plt.xlabel('sqft_living')
plt.ylabel('price')

# Graficar líneas de predicción
for i in range(len(x)):
  plt.plot([x.iloc[i], x.iloc[i]], [y.iloc[i], predt(reg_price,  x.iloc[i])], color='red', linestyle='--')

plt.show()

En el siguiente gráfico, se muestra la línea del modelo de regresión lineal y los coeficientes que se generaron en color rojo para reflejar la distancia correspondiente entre la línea de predicción y el valor real de la observación. Los puntos azules muestran el valor de predicción que brindo el modelo regresión con respecto a la observación o valor real.

In [ ]:
# Seleccionar las variables predictoras y la variable objetivo
# Transformar las variables a logaritmos
y = np.log(casas_prueba['price'])
x = np.log(casas_prueba['sqft_living'])

# Hacer predicciones en el conjunto de prueba
x = sm.add_constant(x)
predicciones = reg_price.predict(x)

# Graficar los puntos y la línea de regresión
plt.scatter(x["sqft_living"], y, color='black')
plt.plot(x["sqft_living"], predicciones, color='red')

# Graficar las líneas verticales y puntos correspondientes
for i in range(len(x)):
    plt.plot([x.iloc[i]["sqft_living"], x.iloc[i]["sqft_living"]], [y.iloc[i], predicciones.iloc[i]], color='green', linestyle='--')
    plt.plot(x.iloc[i]["sqft_living"], predicciones.iloc[i], color='blue', marker='o')

# Agregar etiquetas de los ejes y mostrar la gráfica
plt.xlabel('sqft_living')
plt.ylabel('price')
plt.show()

Se utiliza los datos de prueba para la prediccion

In [ ]:
casas_prueba['prediccion'] = np.exp(reg_price.predict(x))

In [ ]:
print(casas_prueba['prediccion'])

Para una casa de 3000 metros el valor a predecir es de 743361.789 dólares

In [ ]:
# Crear un arreglo con los valores de sqft_living para la nueva casa
sqft_living = 3000

# Realizar la predicción del precio de la casa
precio_casa_pred = np.exp(reg_price.predict([[1, np.log(sqft_living)]]))

# Imprimir el resultado
print(precio_casa_pred)

In [ ]:
casas_prueba['margen'] = casas_prueba['prediccion'] * 0.08
casas_prueba['diferencia'] = casas_prueba['prediccion'] - casas_prueba['price']
casas_prueba['ganancia'] = casas_prueba['diferencia'] - casas_prueba['margen']

Los valores de la predicion de los datos de prueba

In [ ]:
casas_prueba_print = casas_prueba.sort_values('ganancia', ascending=False)

In [ ]:
casas_prueba_print

---
## 🚀 Fase 6 · Despliegue y conclusiones

Estimamos el precio de una casa nueva y guardamos el modelo con `joblib` para reutilizarlo.

📚 Lectura recomendada: [4]

## Resultados

En conclusión, los valores obtenidos por la modelación de regresión lineal simple en los datos se estudiaron pueden establecerse como muy buenos, debido a que el total de las observaciones que puede predecir correctamente es de alrededor del 93 por ciento y solo un 7 por ciento con margen de error. Por lo que se recomienda utilizar el modelo generado para predecir el precio a partir de los metros cuadrados de construcción en el condado de King.

---
## 🚀 Fase 6 · Despliegue y conclusiones — despliegue: predicción interactiva y guardado del modelo

Estimamos el precio de una casa nueva y guardamos el modelo con `joblib` para reutilizarlo.

In [ ]:
#Obtener datos del usuario
# Para pedirlo por teclado use: num = float(input("Digite los pies cuadrados: "))
num = 2500  # pies cuadrados de la casa a valorar

precio_casa = pd.DataFrame({'sqft_living': [num]})
precio_casa['prediccion'] = np.exp(reg_price.predict([[1, np.log(num)]]))
print(f"El precio total de la casa con un margen de error de 8% hacia arriba o hacia abajo es de {precio_casa['prediccion'][0]}")

In [ ]:
#Salvar el modelo
joblib.dump(reg_price, "model.pkl")

In [ ]:
# Cargar el modelo
reg_cargado = joblib.load("model.pkl")

In [ ]:
precio_casa = pd.DataFrame({'sqft_living': [200]})
precio_casa['prediccion'] = np.exp(reg_cargado.predict([[1, np.log(num)]]))
print(f"El precio total de la casa con un margen de error de 8% hacia arriba o hacia abajo es de {precio_casa['prediccion'][0]}")

---
## 🧪 Refuerzo: error del modelo en dólares (no solo en escala logarítmica)

El modelo se ajustó con **logaritmos**, pero quien compra una casa piensa en **dólares**. Por eso evaluamos el error después de revertir con `np.exp`. Una medida muy usada es el **MAPE** (error porcentual absoluto medio): $\text{MAPE}=\frac{100}{n}\sum\left|\frac{y_i-\hat y_i}{y_i}\right|$ [4].

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

real = casas_prueba['price']
est = casas_prueba['prediccion']
print(f'MAE  : ${mean_absolute_error(real, est):,.0f}')
print(f'RMSE : ${np.sqrt(mean_squared_error(real, est)):,.0f}')
print(f'MAPE : {np.mean(np.abs((real - est) / real)) * 100:.1f} %')
print(f'Casas con error menor a 20 %: {np.mean(np.abs((real - est) / real) < 0.20) * 100:.1f} %')

### ✍️ Ejercicios
1. ¿El MAPE es parecido en casas baratas y caras? Calcule el MAPE por cuartil de `price` (`pd.qcut`).
2. Agregue `bedrooms` al modelo (regresión múltiple, como en el Ejemplo 2): ¿baja el MAPE?

---
## ✅ Conclusiones y siguientes pasos

El modelo log-log explica una parte importante de la variación del precio con una sola variable. Para mejorarlo, el siguiente ejemplo incorpora **varias variables predictoras** [4].

---
## 📚 Referencias

Fuentes adicionales para profundizar: [6], [8], [9].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** harlfoxem. (2016). *House Sales in King County, USA* [Conjunto de datos]. Kaggle. https://www.kaggle.com/datasets/harlfoxem/housesalesprediction

**[4]** James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). *An Introduction to Statistical Learning with Applications in Python*. Springer. https://doi.org/10.1007/978-3-031-38747-0

**[5]** Seabold, S., & Perktold, J. (2010). statsmodels: Econometric and statistical modeling with Python. *Proceedings of the 9th Python in Science Conference*, 92–96. https://doi.org/10.25080/Majora-92bf1922-011

**[6]** Spearman, C. (1904). The proof and measurement of association between two things. *The American Journal of Psychology, 15*(1), 72–101. https://doi.org/10.2307/1412159

**[7]** Raschka, S., & Mirjalili, V. (2019). *Python Machine Learning* (3.ª ed.). Packt Publishing. (Figuras usadas desde https://github.com/rasbt/python-machine-learning-book-3rd-edition)

**[8]** McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61. https://doi.org/10.25080/Majora-92bf1922-00a

**[9]** Waskom, M. L. (2021). seaborn: statistical data visualization. *Journal of Open Source Software, 6*(60), 3021. https://doi.org/10.21105/joss.03021